# Criando e testando uma skill propria

Este notebook e um guia pratico + harness de teste para criar uma skill do zero: escrever o conteudo, definir a descricao de ativacao, testar se ela ativa nos momentos certos, e comparar o comportamento do modelo com e sem a skill.

**Skill de exemplo**: resumo de reuniao em um formato interno padronizado. Troque pelo seu proprio caso de uso real na secao 1 -- a estrutura do notebook continua igual.

**Mecanismo usado**: representamos a skill como uma *ferramenta* que o agente pode chamar para carregar as instrucoes completas -- isso espelha como skills funcionam na pratica: uma descricao curta decide a ativacao, e o conteudo completo (que pode ser longo) so e carregado quando realmente necessario. Isso conecta direto com o harness de tool-use construido na Fase 5.

**Sobre os testes**: a avaliacao de ativacao (secoes 1-3) roda aqui com um cliente mock, sem custo de API. A comparacao com/sem skill (secao 4) usa a API real e nao pode ser executada neste ambiente sem internet.

## 1. Escrever a skill

Toda skill tem, no minimo, duas partes:

- **Descricao de ativacao**: uma frase curta que diz *quando* a skill e relevante -- isso e o que decide se ela entra em acao ou nao, entao precisa ser precisa (nem vaga demais, nem restritiva demais)
- **Conteudo**: as instrucoes completas, incluindo formato esperado, regras, e exemplos de bom resultado

Preencha os dois abaixo com o seu proprio caso de uso.

In [ ]:
SKILL_ATIVACAO_DESC = (
    "Use esta skill quando o usuario pedir um resumo de reuniao, ata de reuniao, "
    "ou minuta de encontro de trabalho que precise seguir um formato padronizado."
)

SKILL_CONTEUDO = """# Skill: Resumo de Reuniao (formato interno)

Ao resumir uma reuniao, siga esta estrutura EXATA, nesta ordem:

1. **Participantes**: liste os nomes mencionados
2. **Decisoes tomadas**: lista com marcadores, cada item comecando com verbo no infinitivo
3. **Acoes pendentes**: formato "RESPONSAVEL - ACAO - PRAZO" (prazo pode ser "a definir" se nao mencionado)
4. **Proximos passos**: uma frase curta sobre o que acontece a seguir

Regras:
- Nunca invente nomes, decisoes ou prazos que nao foram mencionados no texto original
- Se a reuniao nao mencionar prazos, marque explicitamente como "a definir", nunca deixe em branco
- Mantenha cada item de "decisoes tomadas" com no maximo uma linha

## Exemplo de bom resultado

Input: "Joao e Maria discutiram o orcamento do projeto X. Decidiram aumentar o budget em 10%. Maria vai enviar a proposta revisada ate sexta."

Output esperado:
**Participantes**: Joao, Maria
**Decisoes tomadas**:
- Aumentar o orcamento do projeto X em 10%
**Acoes pendentes**:
- Maria - enviar proposta revisada - sexta-feira
**Proximos passos**: aguardar envio da proposta revisada por Maria
"""

print("Descricao de ativacao:", SKILL_ATIVACAO_DESC)
print(f"\nConteudo da skill: {len(SKILL_CONTEUDO)} caracteres")


## 2. Representar a skill como ferramenta (progressive disclosure)

Em vez de injetar o conteudo completo da skill em todo prompt (o que gastaria tokens mesmo quando irrelevante), ela vira uma ferramenta que o agente so chama quando a tarefa parece ser da area dela. A **descricao da ferramenta** e exatamente a descricao de ativacao escrita acima -- e isso que o modelo usa para decidir se vale a pena chamar.

In [ ]:
SKILL_TOOL_NAME = "carregar_skill_resumo_reuniao"

SKILL_TOOL_SCHEMA = {
    "name": SKILL_TOOL_NAME,
    "description": SKILL_ATIVACAO_DESC,
    "input_schema": {"type": "object", "properties": {}},
}

TOOL_FUNCTIONS = {
    SKILL_TOOL_NAME: lambda args: {"instrucoes": SKILL_CONTEUDO},
}

def execute_tool(name, args):
    if name not in TOOL_FUNCTIONS:
        return {"error": f"ferramenta desconhecida: {name}"}
    return TOOL_FUNCTIONS[name](args)


## 3. Harness de avaliacao de ativacao

Um dataset de teste com três tipos de caso, de propósito:

- **Casos obvios** que deveriam ativar a skill
- **Um caso de borda**: pede algo parecido (um "resumo"), mas explicitamente fora do escopo da skill (sem formato) -- testa se a descricao de ativacao e precisa o suficiente para nao disparar em falso
- **Casos que nao deveriam ativar**, completamente fora do dominio

O harness reaproveita a estrutura de loop da Fase 5 (`run_agent`), so que agora rastreando se a ferramenta da skill foi chamada ou nao.

In [ ]:
import json

def run_agent(client, messages, max_steps=4, verbose=False):
    steps = 0
    skill_ativada = False
    while steps < max_steps:
        steps += 1
        response = client.create(messages=messages)
        text_blocks = [b for b in response.content if b.type == "text"]
        tool_blocks = [b for b in response.content if b.type == "tool_use"]
        messages.append({"role": "assistant", "content": response.content})

        if any(b.name == SKILL_TOOL_NAME for b in tool_blocks):
            skill_ativada = True

        if response.stop_reason != "tool_use":
            return {
                "resposta": " ".join(b.text for b in text_blocks),
                "skill_ativada": skill_ativada,
                "passos": steps,
            }

        tool_results = []
        for block in tool_blocks:
            result = execute_tool(block.name, block.input)
            tool_results.append({
                "type": "tool_result", "tool_use_id": block.id,
                "content": json.dumps(result, ensure_ascii=False),
            })
        messages.append({"role": "user", "content": tool_results})

    return {"resposta": None, "skill_ativada": skill_ativada, "passos": steps}


test_cases = [
    {"query": "Resuma esta reuniao: Joao e Maria falaram sobre o orcamento.", "deveria_ativar": True},
    {"query": "Pode fazer a ata da reuniao de ontem com o time de vendas?", "deveria_ativar": True},
    {"query": "Preciso de um resumo rapido, informal, so pra mim lembrar depois -- nao precisa formato nenhum.", "deveria_ativar": False},
    {"query": "Qual a capital da Franca?", "deveria_ativar": False},
    {"query": "Escreva um poema sobre o outono.", "deveria_ativar": False},
]


Testamos com um cliente mock -- as respostas aqui sao roteirizadas para simular o que esperamos que um modelo bem instruido faca (ativar nos casos obvios, nao ativar no caso de borda e nos casos fora do dominio). Ao trocar pela API real na secao 4, essas mesmas perguntas podem revelar se o modelo decide diferente do esperado -- o que seria sinal de que a descricao de ativacao precisa de ajuste.

In [ ]:
class FakeToolUseBlock:
    def __init__(self, id, name, input_):
        self.type = "tool_use"; self.id = id; self.name = name; self.input = input_

class FakeTextBlock:
    def __init__(self, text):
        self.type = "text"; self.text = text

class FakeResponse:
    def __init__(self, content, stop_reason):
        self.content = content; self.stop_reason = stop_reason

class FakeAnthropicClient:
    def __init__(self, scripted_responses):
        self.scripted_responses = scripted_responses
        self.call_count = 0
    def create(self, **kwargs):
        r = self.scripted_responses[self.call_count]
        self.call_count += 1
        return r


scripts = [
    [FakeResponse([FakeToolUseBlock("t1", SKILL_TOOL_NAME, {})], "tool_use"),
     FakeResponse([FakeTextBlock("**Participantes**: Joao, Maria...")], "end_turn")],
    [FakeResponse([FakeToolUseBlock("t1", SKILL_TOOL_NAME, {})], "tool_use"),
     FakeResponse([FakeTextBlock("**Participantes**: time de vendas...")], "end_turn")],
    [FakeResponse([FakeTextBlock("Claro, aqui vai um resumo informal e rapido...")], "end_turn")],
    [FakeResponse([FakeTextBlock("A capital da Franca e Paris.")], "end_turn")],
    [FakeResponse([FakeTextBlock("Folhas caem, outono chega...")], "end_turn")],
]

resultados = []
for case, script in zip(test_cases, scripts):
    client = FakeAnthropicClient(script)
    r = run_agent(client, [{"role": "user", "content": case["query"]}])
    ativacao_correta = r["skill_ativada"] == case["deveria_ativar"]
    resultados.append({"query": case["query"], "esperado": case["deveria_ativar"], "ativou": r["skill_ativada"], "correto": ativacao_correta})
    print(f"\'{case['query'][:50]}...\' -> ativou: {r['skill_ativada']} (esperado: {case['deveria_ativar']}) | correto: {ativacao_correta}")

taxa_acerto = sum(r["correto"] for r in resultados) / len(resultados)
print(f"\nTaxa de acerto na decisao de ativacao: {taxa_acerto:.0%}")


## 4. Comparar comportamento com e sem a skill (API real)

O teste mais importante: a skill realmente melhora o resultado? Rodamos a mesma pergunta duas vezes -- uma vez com a ferramenta da skill disponivel, outra vez sem ela -- e comparamos as respostas lado a lado.

**Requer**: `pip install anthropic` e `ANTHROPIC_API_KEY`.

In [ ]:
import os
from anthropic import Anthropic

raw_client = Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

class RealClient:
    def __init__(self, tools):
        self.tools = tools
    def create(self, messages):
        return raw_client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=500,
            tools=self.tools,
            messages=messages,
        )


pergunta_teste = "Resuma esta reuniao: Pedro e Ana discutiram o cronograma do lancamento. Decidiram adiar em 2 semanas. Pedro vai avisar o cliente ate amanha."

print("=== COM a skill disponivel ===")
com_skill = run_agent(RealClient([SKILL_TOOL_SCHEMA]), [{"role": "user", "content": pergunta_teste}])
print(com_skill["resposta"])

print("\n=== SEM a skill disponivel ===")
sem_skill = run_agent(RealClient([]), [{"role": "user", "content": pergunta_teste}])
print(sem_skill["resposta"])


## 5. Checklist de avaliacao qualitativa

Depois de rodar a comparacao acima, avalie as duas respostas com este checklist (adaptavel para a sua propria skill):

- [ ] A resposta "com skill" segue a estrutura exata definida (participantes / decisoes / acoes / proximos passos)?
- [ ] A resposta "sem skill" tem alguma estrutura, ou e so um paragrafo corrido?
- [ ] Algum dos dois inventou informacao que nao estava no texto original (nome, prazo, decisao)?
- [ ] Prazos ausentes foram marcados como "a definir" (com skill) ou simplesmente omitidos (sem skill)?
- [ ] O ganho da skill justifica o custo extra de uma chamada de ferramenta antes da resposta final?

Se a resposta "sem skill" ja for boa o suficiente para o seu caso de uso, isso e um sinal real de que talvez a skill nao seja necessaria -- nem toda tarefa precisa de uma.

## Notas finais

- **Ajustando a descricao de ativacao**: se o teste de ativacao (secao 3) falhar com a API real -- por exemplo, a skill nao ativar no caso de borda esperado, ou ativar onde nao deveria -- o primeiro lugar a ajustar e a `SKILL_ATIVACAO_DESC`, nao o conteudo da skill. Descricoes vagas ("ajuda com reunioes") tendem a sub-ativar ou sobre-ativar; descricoes especificas demais podem deixar de cobrir variacoes legitimas da mesma tarefa
- **Expandir o dataset de teste**: 5 casos so ilustra o mecanismo -- para confianca real na precisao de ativacao, vale um dataset de 20-30 casos, incluindo mais variações de borda
- **Combinando com MCP**: se a skill precisar de acesso a um sistema externo (ex: buscar o template oficial de ata num drive compartilhado, em vez de ter o template fixo no conteudo da skill), essa seria a hora de conectar um servidor MCP correspondente -- a skill continua dizendo *como* fazer, o MCP da *acesso* ao que falta
- **Proximo passo natural**: levar essa mesma skill para dentro do agente com multiplas ferramentas e memoria (Fase 5, projeto 3) -- uma skill raramente vive sozinha, ela compete e coopera com outras ferramentas e skills no mesmo agente
